# Figure 4: Robustness and practical considerations

**Paper:** BATTLE-AMP

**Per-panel stories:**
- **(a)** Most classifiers predict the majority of synthetic sequences as positive, revealing a fundamental lack of specificity even on trivially distinguishable decoys.
- **(b)** Performance varies across peptide length bins, with some models unable to score certain lengths and others showing strong length-dependent biases.
- **(c)** Stricter homology thresholds degrade performance for most models, exposing reliance on sequence similarity to training data rather than learned functional signals.
- **(d)** Computational cost spans orders of magnitude, from seconds (HydrAMP) to hours (SenseXAMP), with peak memory ranging from <1 GB to >48 GB.

**Layout:** 3 rows. Row 1: a (full width, synthetic %pos). Row 2: b (length heatmap) + c (homology heatmap). Row 3: d (full width, runtime + memory).


In [5]:

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# Paths
# ============================================================
CLF_FILE = Path( "../results/aggregated/classification_results.tsv")
RES_FILE = Path("../results/aggregated/resource_usage.tsv")
FIGURE_DIR = Path("../")
FIGURE_DIR.mkdir(exist_ok=True)

# ============================================================
# Constants
# ============================================================
TARGET_DPI = 600
PANEL_W = 6.5

CLF_COLOR  = "#d95f6e"
ACT_COLOR  = "#9467bd"
REG_COLOR  = "#8c8c8c"

GRID_COLOR = "#dddddd"
FONTSIZE_TICK  = 6
FONTSIZE_LABEL = 7
FONTSIZE_PANEL = 10

# ============================================================
# Models
# ============================================================
CLASSIFIERS = [
    "hydramp-amp-classifier", "ampscanner", "amplify",
    "ampeppy", "ampredmfa",
]
ACTIVITY_AWARE = ["hydramp-mic-classifier"]
REGRESSORS = [
    "mbc-attention",
    "apex-ecoli", "apex-saureus", "apex-min",
    "apex-abaumannii", "apex-paeruginosa", "apex-kpneumoniae",
]
ALL_MODELS = CLASSIFIERS + ACTIVITY_AWARE + REGRESSORS

MODEL_DISPLAY = {
    "hydramp-amp-classifier": "HydrAMP$_{AMP}$",
    "ampscanner": "AMP Scanner$_2$",
    "amplify": "AMPlify",
    "ampeppy": "amPEPpy",
    "ampredmfa": "AMPpred-MFA",
    "hydramp-mic-classifier": "HydrAMP$_{MIC}$",
    "mbc-attention": "MBC-Attention",
    "apex-ecoli": "APEX$_{EC}$",
    "apex-saureus": "APEX$_{SA}$",
    "apex-min": "APEX$_{min}$",
    "apex-abaumannii": "APEX$_{AB}$",
    "apex-paeruginosa": "APEX$_{PA}$",
    "apex-kpneumoniae": "APEX$_{KP}$",
}

# Resource panel uses model-level names (multioutput = single run)
RES_DISPLAY = {
    "sensexamp": "SenseXAMP",
    "deep-amp": "Deep-AMP",
    "amplify": "AMPlify",
    "mbc-attention": "MBC-Attention",
    "ampeppy": "amPEPpy",
    "apex": "APEX",
    "ampscanner": "AMP Scanner$_2$",
    "ampredictor": "AMPredictor",
    "mole-amp": "MoLE-AMP",
    "ampredmfa": "AMPpred-MFA",
    "hydramp-amp-classifier": "HydrAMP$_{AMP}$",
    "hydramp-mic-classifier": "HydrAMP$_{MIC}$",
}

RES_TYPE = {
    "sensexamp": "reg", "deep-amp": "reg", "mbc-attention": "reg",
    "apex": "reg", "ampredictor": "reg",
    "amplify": "clf", "ampeppy": "clf", "ampscanner": "clf",
    "ampredmfa": "clf", "mole-amp": "clf",
    "hydramp-amp-classifier": "clf",
    "hydramp-mic-classifier": "act",
}


def model_color(m):
    if m in CLASSIFIERS:
        return CLF_COLOR
    elif m in ACTIVITY_AWARE:
        return ACT_COLOR
    return REG_COLOR


def res_color(m):
    t = RES_TYPE.get(m, "clf")
    if t == "act":
        return ACT_COLOR
    elif t == "reg":
        return REG_COLOR
    return CLF_COLOR


def short_name(m):
    return MODEL_DISPLAY.get(m, m)


# ============================================================
# Load data
# ============================================================
df = pd.read_csv(CLF_FILE, sep="\t")
df = df[df["variant"] != "example-model"].copy()

# Resource data: use battleamp-all dataset for comparable timings
res_df = pd.read_csv(RES_FILE, sep="\t")
res_inf = res_df[
    (res_df["stage"] == "inference") &
    (res_df["dataset"] == "battleamp-all") &
    (res_df["variant"] != "example-model")
].copy()

# ============================================================
# Panel A data: % predicted positive on synthetics
# ============================================================
SYNTH_TASKS = ["synthetic_random", "synthetic_shuffled", "synthetic_realistic"]
SYNTH_LABELS = ["Random", "Shuffled", "Realistic"]

synth_models = [m for m in ALL_MODELS if m in df["variant"].unique()]

def pct_positive(variant, task):
    row = df[(df["variant"] == variant) & (df["task"] == task)]
    if row.empty:
        return np.nan
    r = row.iloc[0]
    if r["n_samples"] > 0:
        return r["n_predicted_positive"] / r["n_samples"] * 100
    return np.nan

# Sort by average % positive ascending (best = lowest on left)
avg_pct = {}
for m in synth_models:
    vals = [pct_positive(m, t) for t in SYNTH_TASKS]
    avg_pct[m] = np.nanmean(vals)
order_a = sorted(synth_models, key=lambda m: avg_pct.get(m, 999))

# ============================================================
# Panel B data: MCC across length bins
# ============================================================
LENGTH_TASKS = ["length_01_10", "length_11_20", "length_21_30", "length_31_50"]
LENGTH_LABELS = ["1-10 aa", "11-20 aa", "21-30 aa", "31-50 aa"]

def safe_val(frame, variant, task, col, min_cov=0.5):
    row = frame[(frame["variant"] == variant) & (frame["task"] == task)]
    if row.empty:
        return np.nan
    r = row.iloc[0]
    if pd.notna(r.get("coverage")) and r["coverage"] < min_cov:
        return np.nan
    val = r[col]
    return float(val) if pd.notna(val) else np.nan

# Order models by type then by average MCC across length bins
heatmap_order = CLASSIFIERS + ACTIVITY_AWARE + REGRESSORS
heatmap_order = [m for m in heatmap_order if m in df["variant"].unique()]

len_mat = np.full((len(heatmap_order), len(LENGTH_TASKS)), np.nan)
for i, m in enumerate(heatmap_order):
    for j, t in enumerate(LENGTH_TASKS):
        len_mat[i, j] = safe_val(df, m, t, "mcc")

# ============================================================
# Panel C data: MCC across homology bins
# ============================================================
HOM_TASKS = ["homology_40", "homology_60", "homology_80", "broad_activity"]
HOM_LABELS = ["40%", "60%", "80%", "Full"]

hom_mat = np.full((len(heatmap_order), len(HOM_TASKS)), np.nan)
for i, m in enumerate(heatmap_order):
    for j, t in enumerate(HOM_TASKS):
        hom_mat[i, j] = safe_val(df, m, t, "mcc")

# ============================================================
# Panel D data: runtime and memory
# ============================================================
res_order = res_inf.sort_values("s", ascending=True)["variant"].tolist()
res_order = [m for m in res_order if m in RES_DISPLAY]

# ============================================================
# Global style
# ============================================================
matplotlib.rcParams["font.family"] = "sans-serif"
matplotlib.rcParams["font.sans-serif"] = [
    "Helvetica", "Arial", "Liberation Sans", "DejaVu Sans",
]
matplotlib.rcParams["mathtext.default"] = "regular"
matplotlib.rcParams["axes.linewidth"] = 0.5
matplotlib.rcParams["xtick.major.width"] = 0.4
matplotlib.rcParams["ytick.major.width"] = 0.4


def style_ax(ax, ylabel=None, xlabel=None):
    ax.set_facecolor("white")
    ax.yaxis.grid(True, color=GRID_COLOR, linewidth=0.4)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", labelsize=FONTSIZE_TICK, length=2)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=FONTSIZE_LABEL)
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=FONTSIZE_LABEL)




In [6]:
# ============================================================
# Build figure
# ============================================================
fig = plt.figure(figsize=(6.1, 7.8), dpi=TARGET_DPI, facecolor="white")

gs = gridspec.GridSpec(
    3, 2, hspace=0.50, wspace=0.08,
    height_ratios=[0.8, 1.0, 0.7],
    left=0.12, right=0.90, top=0.975, bottom=0.06,
)

ax_a = fig.add_subplot(gs[0, :])

# Heatmaps: left = length, right = homology
ax_b = fig.add_subplot(gs[1, 0])
ax_c = fig.add_subplot(gs[1, 1])

ax_d = fig.add_subplot(gs[2, :])


# ================================================================
# Panel A: % predicted positive on synthetic datasets
# ================================================================
n_a = len(order_a)
xa = np.arange(n_a)
bar_w = 0.26
offsets = [-bar_w, 0, bar_w]
alphas = [0.35, 0.70, 1.0]

for k, (task, label, alpha) in enumerate(
    zip(SYNTH_TASKS, SYNTH_LABELS, alphas)
):
    vals = [pct_positive(m, task) for m in order_a]
    ax_a.bar(
        xa + offsets[k], vals, bar_w,
        color=[model_color(m) for m in order_a],
        alpha=alpha, edgecolor="#aaaaaa", linewidth=0.3,
    )

ax_a.set_xticks(xa)
labels_a = ax_a.set_xticklabels(
    [short_name(m) for m in order_a],
    rotation=55, ha="right", fontsize=6,
)
for lbl, m in zip(labels_a, order_a):
    lbl.set_color(model_color(m))
ax_a.set_xlim(-0.6, n_a - 0.4)
ax_a.set_ylim(0, 105)
style_ax(ax_a, ylabel="Predicted positive (%)")

# Legend for synthetic tasks
legend_handles = [
    Patch(facecolor="#888888", alpha=a, edgecolor="#aaaaaa", label=l)
    for a, l in zip(alphas, SYNTH_LABELS)
]
ax_a.legend(
    handles=legend_handles, fontsize=6, loc="upper left",
    frameon=True, facecolor="white", edgecolor="#cccccc",
    handlelength=1.0, handleheight=0.7,
)

# Type legend
type_handles = [
    Patch(facecolor=CLF_COLOR, edgecolor="none", label="Classifier"),
    Patch(facecolor=ACT_COLOR, edgecolor="none", label="HydrAMP$_{MIC}$"),
    Patch(facecolor=REG_COLOR, edgecolor="none", label="Regressor"),
]
ax_a.legend(
    handles=legend_handles + type_handles,
    fontsize=6, loc="upper left", frameon=True,
    facecolor="white", edgecolor="#cccccc", ncol=2,
    handlelength=1.0, handleheight=0.7, columnspacing=0.8,
)


# ================================================================
# Panel B: MCC heatmap across length bins
# ================================================================
cmap = matplotlib.colormaps["RdYlGn"]
cmap.set_bad("#e8e8e8")
norm = mcolors.TwoSlopeNorm(vmin=-0.1, vcenter=0.3, vmax=0.7)

im_b = ax_b.imshow(
    len_mat, aspect="auto", cmap=cmap, norm=norm,
    interpolation="nearest",
)

# Annotate cells
for i in range(len_mat.shape[0]):
    for j in range(len_mat.shape[1]):
        val = len_mat[i, j]
        if np.isnan(val):
            ax_b.text(j, i, "-", ha="center", va="center",
                      fontsize=6, color="#999999")
        else:
            color = "white" if val < 0.1 else "black"
            ax_b.text(j, i, f"{val:.2f}", ha="center", va="center",
                      fontsize=5.5, color=color)

ax_b.set_xticks(np.arange(len(LENGTH_LABELS)))
ax_b.set_xticklabels(LENGTH_LABELS, fontsize=6)
ax_b.set_yticks(np.arange(len(heatmap_order)))
ylabels_b = ax_b.set_yticklabels(
    [short_name(m) for m in heatmap_order], fontsize=6,
)
for lbl, m in zip(ylabels_b, heatmap_order):
    lbl.set_color(model_color(m))

ax_b.tick_params(axis="both", length=0)
ax_b.set_title("MCC by peptide length", fontsize=7, pad=4, color="#555555")

# Group separator line between classifiers+activity and regressors
sep_y = len(CLASSIFIERS) + len(ACTIVITY_AWARE) - 0.5
ax_b.axhline(sep_y, color="black", linewidth=0.8)


# ================================================================
# Panel C: MCC heatmap across homology bins
# ================================================================
im_c = ax_c.imshow(
    hom_mat, aspect="auto", cmap=cmap, norm=norm,
    interpolation="nearest",
)

for i in range(hom_mat.shape[0]):
    for j in range(hom_mat.shape[1]):
        val = hom_mat[i, j]
        if np.isnan(val):
            ax_c.text(j, i, "-", ha="center", va="center",
                      fontsize=6, color="#999999")
        else:
            color = "white" if val < 0.1 else "black"
            ax_c.text(j, i, f"{val:.2f}", ha="center", va="center",
                      fontsize=5.5, color=color)

ax_c.set_xticks(np.arange(len(HOM_LABELS)))
ax_c.set_xticklabels(HOM_LABELS, fontsize=6)
ax_c.set_yticks(np.arange(len(heatmap_order)))
ax_c.set_yticklabels([])  # shared y-axis with panel b
ax_c.tick_params(axis="both", length=0)
ax_c.set_title("MCC by max homology to training data",
                fontsize=7, pad=4, color="#555555")
ax_c.axhline(sep_y, color="black", linewidth=0.8)

# Shared colorbar for both heatmaps
cbar_ax = fig.add_axes([0.91, gs[1, 0].get_position(fig).y0,
                         0.012,
                         gs[1, 0].get_position(fig).height])
cbar = fig.colorbar(im_b, cax=cbar_ax)
cbar.ax.tick_params(labelsize=6)
cbar.set_label("MCC", fontsize=7)


# ================================================================
# Panel D: Runtime + Memory (horizontal bars, log scale)
# ================================================================
n_d = len(res_order)
yd = np.arange(n_d)

times = [res_inf[res_inf["variant"] == m]["s"].values[0] for m in res_order]
mem_gb = [res_inf[res_inf["variant"] == m]["max_rss"].values[0] / 1024
          for m in res_order]
colors_d = [res_color(m) for m in res_order]

bar_h = 0.38

# Time bars (bottom axis)
ax_d.barh(yd + bar_h/2, times, bar_h,
          color=colors_d, alpha=0.8, edgecolor="#aaaaaa", linewidth=0.3)
ax_d.set_xscale("log")
ax_d.set_xlabel("Wall time (s)", fontsize=FONTSIZE_LABEL)

# Memory bars (top axis)
ax_d2 = ax_d.twiny()
ax_d2.barh(yd - bar_h/2, mem_gb, bar_h,
           color=colors_d, alpha=0.35, edgecolor="#aaaaaa", linewidth=0.3)
ax_d2.set_xscale("log")
ax_d2.set_xlabel("Peak memory (GB)", fontsize=FONTSIZE_LABEL, color="#888888")
ax_d2.tick_params(axis="x", labelsize=6, colors="#888888")
ax_d2.spines["top"].set_visible(True)
ax_d2.spines["top"].set_color("#cccccc")

ax_d.set_yticks(yd)
ylabels_d = ax_d.set_yticklabels(
    [RES_DISPLAY.get(m, m) for m in res_order], fontsize=6,
)
for lbl, m in zip(ylabels_d, res_order):
    lbl.set_color(res_color(m))

ax_d.set_ylim(-0.6, n_d - 0.4)
ax_d.invert_yaxis()
ax_d.spines["top"].set_visible(False)
ax_d.spines["right"].set_visible(False)
ax_d.tick_params(axis="both", labelsize=FONTSIZE_TICK, length=2)
ax_d.xaxis.grid(True, color=GRID_COLOR, linewidth=0.4)
ax_d.set_axisbelow(True)

# Legend for time vs memory
ax_d.legend(
    [Patch(facecolor="#888888", alpha=0.8, edgecolor="#aaaaaa"),
     Patch(facecolor="#888888", alpha=0.35, edgecolor="#aaaaaa")],
    ["Wall time", "Peak memory"],
    fontsize=6, loc="lower right", frameon=True,
    facecolor="white", edgecolor="#cccccc",
    handlelength=1.0, handleheight=0.7,
)


# ================================================================
# Panel labels
# ================================================================
for label, ax in [("a", ax_a), ("b", ax_b), ("c", ax_c), ("d", ax_d)]:
    pos = ax.get_position()
    fig.text(pos.x0 - 0.07, pos.y1 + 0.005, label,
             fontsize=FONTSIZE_PANEL, fontweight="bold",
             va="bottom", ha="left")


# ================================================================
# Save
# ================================================================
out_pdf = FIGURE_DIR / "figure4_robustness.pdf"
out_png = FIGURE_DIR / "figure4_robustness.png"
fig.savefig(str(out_pdf), bbox_inches="tight", pad_inches=0.02,
            dpi=TARGET_DPI, facecolor="white")
fig.savefig(str(out_png), bbox_inches="tight", pad_inches=0.02,
            dpi=TARGET_DPI, facecolor="white")
print(f"\nSaved {out_pdf} and {out_png}")
plt.close(fig)



Saved ../figure4_robustness.pdf and ../figure4_robustness.png


## Alt text

**Figure 4.** Robustness evaluation and computational costs of AMP prediction models.
Models color-coded: classifiers (rose), HydrAMP-MIC (purple), regressors (grey).
**(a)** Percentage of synthetic sequences predicted as positive on three decoy datasets
(Random, Shuffled, Realistic), sorted by average false-positive rate ascending.
APEX species regressors predict <5% positives across all datasets; most classifiers
exceed 80% on Shuffled and Realistic sequences.
**(b)** MCC heatmap across four peptide length bins (1-10, 11-20, 21-30, 31-50 aa).
Grey cells indicate coverage below 50% (model cannot score those lengths). MBC-Attention
is stable across all bins (~0.65). HydrAMP models cannot process peptides >30 aa.
APEX species-specific models perform best on short peptides and degrade on longer ones.
**(c)** MCC heatmap across three maximum-homology thresholds (40%, 60%, 80%) and the
full GeneralActivity dataset. Most models show degradation at stricter thresholds.
MBC-Attention improves slightly at lower homology (0.68 at 40% vs 0.63 on full),
suggesting genuine generalization rather than memorization.
**(d)** Inference wall time (dark bars, bottom axis, log scale) and peak memory
(light bars, top axis, log scale) on the full benchmark dataset. SenseXAMP requires
~2 hours and ~6 GB; MoLE-AMP and AMPredictor peak at 48 and 33 GB respectively;
HydrAMP models complete in under 90 seconds with <1 GB memory.
